#### Read the data and load into the dataframe and flattern the data in the given output format
**DBFS Path:** /Volumes/spark_workouts/customerdb/usecasefile/student_sport.csv
#### Input CSV
studid,studname,studsports <br>
1001,"Raghav","{'sportsname':'cricket','year':2024,'sportcoach':'Kamalesh'}" <br>
1002,"Madhav","{'sportsname':'football','year':2025,'sportcoach':'Rajesh'}" <br>
1003,"Kannan","{'sportsname':'hockey','year':2025,'sportcoach':'Vimal'}" <br>

#### Output
| studid | studname | sportsname | year | sportcoach |
|--------|----------|------------|------|------------|
| 1001 | Raghav | cricket | 2024 | Kamalesh |
| 1002 | Madhav | football| 2025 | Rajesh |
| 1003 | Kannan | hockey |  2025 | Vimal |


In [0]:
# Method 1: Using from_json() with user defined JSON schema
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import from_json

json_schema = StructType([
    StructField('sportsname', StringType(), True),
    StructField('year', IntegerType(), True),
    StructField('sportcoach', StringType(), True)
])

df = spark.read.format('csv').options(header=True, inferSchema=True).load('/Volumes/spark_workouts/customerdb/usecasefile/student_sport.csv')
df1 = df.withColumn('studsports', from_json(df.studsports, json_schema))
df2 = df1.select(
    "studid", 
    "studname", 
    df1.studsports.sportsname.alias('sportsname'), 
    df1.studsports.year.alias('year'), 
    df1.studsports.sportcoach.alias('sportcoach')
    )
display(df2)

In [0]:
# Method 2: Using from_json() with inferring JSON schema
from pyspark.sql.functions import schema_of_json, from_json

df = spark.read.format('csv').options(header=True, inferSchema=True).load('/Volumes/spark_workouts/customerdb/usecasefile/student_sport.csv')

sample_json = df.select(df.studsports).first()[0]
json_schema_1 = schema_of_json(sample_json)
df1 = df.withColumn('studsports', from_json(df.studsports, json_schema_1))
df2 = df1.select(
    "studid", 
    "studname", 
    df1.studsports.sportsname.alias('sportsname'), 
    df1.studsports.year.alias('year'), 
    df1.studsports.sportcoach.alias('sportcoach')
    )
display(df2)

In [0]:
# Method 3: Using get_json_object() - No need of predefined JSON schema
from pyspark.sql.functions import get_json_object

df = spark.read.format('csv').options(header=True, inferSchema=True).load('/Volumes/spark_workouts/customerdb/usecasefile/student_sport.csv')

df1 = df.select(
    "studid", 
    "studname",
    get_json_object(df.studsports, "$.sportsname").alias('sportsname'),
    get_json_object(df.studsports, "$.year").alias('year'),
    get_json_object(df.studsports, "$.sportcoach").alias('sportcoach')
    )
display(df1)

In [0]:
# Method 4: Using json_tuple()
from pyspark.sql.functions import json_tuple

df = spark.read.format('csv').options(header=True, inferSchema=True).load('/Volumes/spark_workouts/customerdb/usecasefile/student_sport.csv')

df1 = df.select(
    "studid", 
    "studname",
    json_tuple(df.studsports, "sportsname", "year", "sportcoach").alias("sportsname", "year", "sportcoach")
    )
display(df1)